# Lab 4 — Production Readiness & Project Launch
**Block 4 · Production & Homework Launch · PGE5 / M2**

This final lab assembles Labs 1–3 into a complete production system:
monitoring, versioning, EU AI Act — then launches the final project you will submit
in 2 days.

### Objectives
1. **Monitoring**: 5 metrics that detect 90% of production failures.
2. **Versioning**: hash the system prompt to trace every behaviour change.
3. **EU AI Act**: identify your agent's risk tier and the associated obligation.
4. **Production checklist**: validate your agent before the first real user.
5. **Final project**: choose your topic, write the problem statement, sketch the architecture.

> **Offline mode.** All cells run without an API key.


## 0. Setup

In [ ]:
from llm_helpers import (
    make_client, credentials_available, ToolRegistry, tool_schema, run_agent,
)
import hashlib, time, json, re
from collections import defaultdict

ONLINE = credentials_available()
print('Mode:', '🌐 ONLINE' if ONLINE else '⚙️  OFFLINE (mock)')

def demo(script=None):
    return make_client(offline_script=script, quiet=True)


## 1. Evaluation — exact-match, LLM-judge, per-category

In [ ]:
TEST_CASES = [
    {'question':  'How many people are displaced per year according to UNHCR?',
     'reference': '21.5 million', 'category': 'figures'},
    {'question':  'What does the World Bank project for 2050?',
     'reference': '216 million', 'category': 'projections'},
    {'question':  'Which Southeast Asian countries are most exposed?',
     'reference': 'Bangladesh, Vietnam, Philippines', 'category': 'geography'},
    {'question':  'Increase in displacement in the Philippines?',
     'reference': '15%', 'category': 'figures'},
    {'question':  'What does the Mekong Delta risk?',
     'reference': '40% of floodable land', 'category': 'risks'},
]

def agent_production(q, script=None):
    client = demo(script or [f'[production response] {q[:60]}'])
    resp = client.complete([
        {'role':'system','content':'Answer from the context. CONFIDENCE: HIGH/MEDIUM/LOW.'},
        {'role':'user',  'content': q},
    ])
    return resp.content

def exact_match(resp, ref):
    return any(m.lower() in resp.lower() for m in ref.split() if len(m) > 3)

print('=== 1.1 EXACT-MATCH ===')
scores_em = []
for case in TEST_CASES:
    resp = agent_production(case['question'], script=[f"[response] {case['reference']}"])
    ok  = exact_match(resp, case['reference'])
    scores_em.append(ok)
    print(f"  {'✓' if ok else '✗'} [{case['category']}] {case['question'][:50]}…")
print(f'\nScore: {sum(scores_em)}/{len(scores_em)}')


In [ ]:
def llm_judge(question, response, reference, script=None):
    client = demo(script or [{'final': json.dumps({'score':8,'justification':'Correct.'})}])
    msg = [{'role':'user','content':
        f'Question: {question}\nResponse: {response}\nReference: {reference}\n'
        'Rate out of 10. JSON only: {"score":<0-10>,"justification":"<sentence>"}'}]
    raw = client.complete(msg).content or '{}'
    m = re.search(r'\{.*?\}', raw, re.DOTALL)
    if m:
        try: return json.loads(m.group())
        except: pass
    return {'score': 0, 'justification': 'parse error'}

print('=== 1.2 LLM-JUDGE ===')
for case in TEST_CASES[:3]:
    resp = agent_production(case['question'], script=[case['reference']])
    ev  = llm_judge(case['question'], resp, case['reference'])
    print(f"  Score {ev['score']}/10 — {ev['justification'][:60]}")


In [ ]:
print('=== 1.3 PER CATEGORY ===')
by_cat = defaultdict(list)
for case, ok in zip(TEST_CASES, scores_em):
    by_cat[case['category']].append(ok)
for cat, res in sorted(by_cat.items()):
    rate = sum(res)/len(res)
    bar = '█'*int(rate*10) + '░'*(10-int(rate*10))
    print(f'  {cat:<15} {bar} {rate:.0%}  ({sum(res)}/{len(res)})')


## 2. Production monitoring

In [ ]:
class AgentMonitor:
    def __init__(self):
        self.n_runs=0; self.total_cost=0.0; self.alerts=[]
        self.tools = defaultdict(lambda: {'calls':0,'errors':0,'ms_total':0})

    def record_run(self, question, response, duration_s, cost_usd):
        self.n_runs+=1; self.total_cost+=cost_usd
        if duration_s > 60:  self._alert(f'SLOW RUN: {duration_s:.1f}s')
        if cost_usd>.50:     self._alert(f'EXPENSIVE RUN: ${cost_usd:.4f}')
        if not response or len(response)<20: self._alert(f'EMPTY RESPONSE: {question[:40]}')

    def record_tool(self, name, success, duration_ms):
        self.tools[name]['calls']+=1; self.tools[name]['ms_total']+=duration_ms
        if not success:
            self.tools[name]['errors']+=1
            rate = self.tools[name]['errors']/self.tools[name]['calls']
            if rate > 0.20: self._alert(f'ERRORS {name}: {rate:.0%}')

    def _alert(self, msg): self.alerts.append(msg); print(f'⚠  {msg}')

    def report(self):
        print(f'\nRuns: {self.n_runs} | Cost: ${self.total_cost:.4f}')
        for name,s in self.tools.items():
            avg = s['ms_total']/s['calls'] if s['calls'] else 0
            err = s['errors']/s['calls']  if s['calls'] else 0
            print(f'  {name:<20} {s["calls"]} calls  {err:.0%} errors  {avg:.0f}ms avg')

mon = AgentMonitor()
for i in range(5):
    mon.record_run(f'question {i+1}', 'response '*5, 5.+i*2, .003*(i+1))
    mon.record_tool('search_knowledge', i%3!=2, 300+i*50)
mon.report()


## 3. Versioning — hash the system prompt

In [ ]:
def hash_prompt(prompt):
    return hashlib.sha256(prompt.encode()).hexdigest()[:12]

SYSTEM_PROMPT = 'TODO: paste your real system prompt here.'

AGENT_VERSION = {
    'version':       '1.0.0',
    'system_prompt': hash_prompt(SYSTEM_PROMPT),
    'model':         'gpt-4o-mini',
    'tools':         ['search_knowledge','recall_memory','store_finding'],
    'created_at':    time.strftime('%Y-%m-%dT%H:%M:%S'),
}
for k,v in AGENT_VERSION.items(): print(f'  {k:<20} {v}')
print('\n→ If the prompt hash changes, behaviour changed — trace it.')


## 4. EU AI Act — risk tier

In [ ]:
def risk_tier(description):
    d = description.lower()
    if any(m in d for m in ['social scoring','biometric surveillance']):
        return 'PROHIBITED', 'Do not deploy.'
    if any(m in d for m in ['hiring','credit','justice','police','migration']):
        return 'HIGH RISK', 'HITL + audit trail + conformity assessment.'
    if any(m in d for m in ['chatbot','assistant','research','summary']):
        return 'LIMITED RISK', 'Inform users (interaction with AI).'
    return 'MINIMAL RISK', 'No obligation.'

# TODO: describe your agent
desc = 'Document research agent on climate displacement, internal use.'
tier, obligation = risk_tier(desc)
print(f'Description: {desc}')
print(f'EU AI Act tier: {tier}')
print(f'Obligation: {obligation}')
print('\n→ Add this paragraph to the EU AI Act section of your REPORT.md.')


### 📝 Questions on the outputs above (Parts 1–4)

Your exact numbers depend on the model and change between runs — answer from **your** output.

**Q1 — Three methods disagree (Parts 1.1 & 1.2).** Exact-match scored some answers 0 that the
LLM-judge scored 6–8. Pick one answer they disagreed on and read it. Which method is right —
or is the question itself wrong? What does exact-match actually measure, versus what the
LLM-judge measures?

**Q2 — Trust the category chart? (Part 1.3).** The per-category chart shows "projections 0%".
But that percentage is computed from *which* of the two scoring methods? Would the chart look
different if it used the LLM-judge scores instead? What does this teach you about reading an
evaluation dashboard?

**Q3 — When to use which (Part 1).** Exact-match is instant and free; the LLM-judge costs an
extra model call per answer and varies between runs. For each, name one situation where it is
the *right* choice and one where it is dangerous to rely on.

**Q4 — The monitoring alert (Part 2).** An alert fired for `search_knowledge` error rate. What
threshold triggered it, and why is a *per-tool* error rate more useful than a single overall
"agent works / doesn't work" flag? What real failure would this catch that a user might never
report?

**Q5 — Why hash the prompt? (Part 3).** Your version shows a `system_prompt` hash. Suppose next
week the agent starts giving worse answers. How does having this hash in every log entry help you
find the cause faster than reading the prompt by eye?

**Q6 — Risk tier judgement (Part 4).** The agent was classified LIMITED RISK. Describe one change
to *what the agent does* that would push it into HIGH RISK. What new obligation would that trigger,
and why does the regulation treat those cases differently?

> *Discuss in pairs (5 min), then compare across the models people ran.*

## 5. Generating the project skeleton

In [ ]:
# Generate the project files
import os
os.makedirs('project', exist_ok=True)

# agent.py — skeleton
with open('project/agent.py', 'w') as f:
    f.write('# agent.py\n'
            'from llm_helpers import make_client, ToolRegistry, tool_schema, run_agent\n'
            '# from retriever import production_retrieve  # Lab 1\n'
            '# from guardrails import l1_filter, l4_gate  # Lab 2\n'
            '# from reasoning import self_consistent_answer  # Lab 3\n\n'
            'SYSTEM = """\n'
            'TODO: paste your system prompt here (with few-shot CoT).\n'
            '\"\"\"\n\n'
            'def run(question):\n'
            '    client = make_client()\n'
            '    registry = ToolRegistry()\n'
            '    history = [{"role": "system", "content": SYSTEM},\n'
            '               {"role": "user",   "content": question}]\n'
            '    return run_agent(client, registry, history)[-1].get("content", "")\n')

# .env.example
with open('project/.env.example', 'w') as f:
    f.write('LLM_PROVIDER=openai\nLLM_MODEL=gpt-4o-mini\nOPENAI_API_KEY=\n')

print('Files generated in project/')
for fn in ['agent.py', '.env.example']:
    print(f'  project/{fn}')


In [ ]:
# REPORT.md — template
REPORT_TEMPLATE = """# REPORT.md

## 1. Problem statement
Who is the user? Concrete scenario. What the agent does that a chatbot cannot.

## 2. Architecture
Components, diagram, one non-obvious design decision.

## 3. RAGAS evaluation
| Metric | Baseline | Improved |
|---|---|---|
| context_recall | | |
| context_precision | | |
| faithfulness | | |
| answer_relevancy | | |

## 4. Security — before/after table (5 injection tests)

## 5. EU AI Act — risk tier + implemented obligation

## 6. Limitations and next step

## 7. AI use declaration
| Component | Human | AI-assisted | AI-generated |
|---|---|---|---|
"""

with open('project/REPORT.md', 'w') as f:
    f.write(REPORT_TEMPLATE)
print('project/REPORT.md generated')


## 6. Design session — before you leave

In [ ]:
# ── DESIGN WORKSPACE ──────────────────────────────────────────────────────

GROUP = ['Name 1', 'Name 2']  # TODO: fill in
TOPIC = 'TODO: your topic'    # TODO: list in HW_brief.md

PEAS = {
    'Performance':  ['TODO: measurable criterion 1', 'TODO: failure mode to exclude'],
    'Environment':  ['TODO: data sources', 'TODO: APIs'],
    'Actuators':    ['TODO: tool 1', 'TODO: tool 2', 'TODO: MCP server'],
    'Sensors':      ['TODO: input', 'TODO: what the agent reads'],
}

print(f'Group: {GROUP}')
print(f'Topic: {TOPIC}')
print()
for comp, items in PEAS.items():
    print(f'  {comp}:')
    for item in items: print(f'    • {item}')


In [ ]:
# Final check — to show the instructor
project_items = [
    ('Group registered',           len(GROUP)>=2 and GROUP[0]!='Name 1'),
    ('Topic chosen',               TOPIC!='TODO: your topic'),
    ('PEAS started',               any('TODO' not in v for vl in PEAS.values() for v in vl)),
    ('GitHub repo created',        False),  # TODO: check when done
    ('Blocks D1/D2/D3 validated',  False),  # TODO: check when done
]
print('PROJECT CHECKLIST:')
for text, done in project_items:
    print(f"  {'✓' if done else '☐'}  {text}")
print()
print('Deadline: 2 days.')
print('Commits after that do not count.')


## 7. Production checklist

In [ ]:
CHECKLIST = {
    'Security':     ['L1: 5/5 injection tests passed',
                     'L4: risk gate for all tools',
                     'Token budget: max_usd + recording'],
    'Reliability':  ['Tools return strings (no exceptions)',
                     'max_steps in the loop'],
    'Retrieval':    ['Hybrid BM25+TF-IDF+RRF',
                     'Cross-encoder reranking',
                     'RAGAS baseline + improved documented'],
    'Reasoning':    ['Few-shot CoT (EVIDENCE/ANALYSIS/CONCLUSION/CONFIDENCE)',
                     'Self-Consistency k≥3'],
    'Compliance':   ['EU AI Act tier documented',
                     'User disclosure if limited risk'],
}
total = done = 0
for cat, items in CHECKLIST.items():
    print(f'\n{cat}:')
    for item in items:
        total+=1
        print(f'  ☐  {item}')  # TODO: mark ✓ the completed items
print(f'\nComplete the boxes before submission.')


## ✅ Module recap

What you can now build:

| Component | Lab | Technique |
|---|---|---|
| Production retriever | 1 | TF-IDF → BM25 → hybrid RRF → cross-encoder |
| Quality measurement | 1 | hit@k / MRR proxy → RAGAS |
| Input filter | 2 | L1: Unicode + injection patterns |
| Action control | 2 | L4: SAFE / MONITOR / CONFIRM / BLOCK |
| Token budget | 2 | Hard cap per run |
| Reasoning | 3 | Zero-shot CoT → few-shot → Self-Consistency |
| Monitoring | 4 | Latency, cost, error rate per tool |
| Versioning | 4 | System-prompt hash |
| Compliance | 4 | EU AI Act — tier + obligation |

**Good luck with the project. Excellence comes from the clarity of the problem, not the number of features.**
